Olá, Valter

Meu nome é Ramon. Ao ao longo do texto farei algumas observações sobre melhorias no código. Nas partes em que não houver comentários, está tudo certo, ok? Estarei aberto a feedbacks e discussões sobre o tema.

**Peço que mantenha e não altere os comentários que eu fizer por aqui para que possamos nos localizar posteriormente, ok?**

Mais uma coisa, vamos utilizar um código de cores para você entender os meus feedbacks no seu notebook. Funciona assim:

<div class="alert alert-block alert-success">
<b> Comentário do revisor: </b> <a class="tocSkip"></a>

Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor: </b> <a class="tocSkip"></a>

Alerta não crítco, mas que pode ser corrigido para melhoria geral no seu código/análise
</div>

<div class="alert alert-block alert-danger">

<b>Comentário do revisor: </b> <a class="tocSkip"></a>
    
Erro que precisa ser consertado, caso contrário seu projeto **não** será aceito.
</div>

Você pode me responder usando isso:
<div class="alert alert-block alert-info">
<b>Resposta do Aluno.</b> <a class="tocSkip"></a>
</div>

###


<div class="alert alert-block alert-success">
<b>Comentário Geral do Revisor</b> <a class="tocSkip"></a>

Obrigado por enviar seu projeto. 
    
Parabéns, você fez um bom trabalho.

Como pontos fortes do projeto, destaco:
- Código simples e eficiente (sem loop desnecessário)
- Uso adequado dos recursos da lição
- Calibração de hiperparâmetros correta
  
Gabaritou o projeto. Parabéns. Desejo sucesso na jornada. 

</div>




# Sprint 8 — Recomendação de Planos Megaline

## Objetivo
A operadora Megaline pretende migrar os seus clientes dos planos antigos para os novos planos **Smart** ou **Ultra**. O objetivo deste projeto é desenvolver um modelo de classificação que, com base no comportamento mensal do utilizador, recomende o plano mais adequado.

**Meta de acurácia:** ≥ 0.75

## Estrutura do projeto
1. Carregamento e análise dos dados
2. Divisão dos dados (treino / validação / teste)
3. Treino e comparação de modelos
4. Avaliação no conjunto de teste
5. Prova real (sanity check)

## 1. Carregamento e análise dos dados

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Carregar os dados
df = pd.read_csv('/datasets/users_behavior.csv')

print('Dimensões do dataset:', df.shape)
df.head()

Dimensões do dataset: (3214, 5)


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [15]:
# Informações gerais
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [16]:
# Estatísticas descritivas
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [17]:
# Verificar valores ausentes
print('Valores ausentes por coluna:')
print(df.isnull().sum())

Valores ausentes por coluna:
calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64


In [18]:
# Distribuição da variável alvo
print('Distribuição da variável alvo (is_ultra):')
print(df['is_ultra'].value_counts())
print()
print('Proporção:')
print(df['is_ultra'].value_counts(normalize=True).round(3))

Distribuição da variável alvo (is_ultra):
0    2229
1     985
Name: is_ultra, dtype: int64

Proporção:
0    0.694
1    0.306
Name: is_ultra, dtype: float64


### Conclusões — Análise dos dados

- O dataset não apresenta valores ausentes, pelo que não é necessário pré-processamento adicional.
- As features são todas numéricas: `calls`, `minutes`, `messages`, `mb_used`.
- A variável alvo `is_ultra` é binária: **0 = Smart**, **1 = Ultra**.
- O dataset está ligeiramente desbalanceado (mais clientes no plano Smart), o que é comum em dados reais.

## 2. Divisão dos dados

Dividimos os dados em três conjuntos:
- **Treino (60%)** — para ajustar os modelos
- **Validação (20%)** — para comparar e selecionar o melhor modelo
- **Teste (20%)** — avaliação final, usada apenas uma vez

Esta divisão garante que o modelo é avaliado em dados que nunca viu durante o treino.

In [19]:
# Separar features e target
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# Primeiro split: 60% treino, 40% temporário
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.40, random_state=12345
)

# Segundo split: dividir o temporário em 50/50 → 20% validação, 20% teste
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.50, random_state=12345
)

print(f'Treino:    {features_train.shape[0]} amostras ({features_train.shape[0]/len(df):.0%})')
print(f'Validação: {features_valid.shape[0]} amostras ({features_valid.shape[0]/len(df):.0%})')
print(f'Teste:     {features_test.shape[0]} amostras ({features_test.shape[0]/len(df):.0%})')

Treino:    1928 amostras (60%)
Validação: 643 amostras (20%)
Teste:     643 amostras (20%)


## 3. Treino e comparação de modelos

Vamos testar três modelos de classificação, variando os seus hiperparâmetros, e comparar a acurácia no conjunto de validação:

1. **Árvore de Decisão** — modelo simples e interpretável
2. **Random Forest** — conjunto de árvores, geralmente mais robusto
3. **Regressão Logística** — modelo linear de referência (baseline)

### 3.1 Árvore de Decisão

In [20]:
best_dt_accuracy = 0
best_dt_depth = 0

for depth in range(1, 11):
    model_dt = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    model_dt.fit(features_train, target_train)
    predictions_valid = model_dt.predict(features_valid)
    accuracy = accuracy_score(target_valid, predictions_valid)
    print(f'max_depth={depth:2d} | Acurácia (validação): {accuracy:.4f}')
    if accuracy > best_dt_accuracy:
        best_dt_accuracy = accuracy
        best_dt_depth = depth
        best_dt_model = model_dt

print(f'\nMelhor Árvore de Decisão: max_depth={best_dt_depth} | Acurácia={best_dt_accuracy:.4f}')

max_depth= 1 | Acurácia (validação): 0.7543
max_depth= 2 | Acurácia (validação): 0.7823
max_depth= 3 | Acurácia (validação): 0.7854
max_depth= 4 | Acurácia (validação): 0.7792
max_depth= 5 | Acurácia (validação): 0.7792
max_depth= 6 | Acurácia (validação): 0.7838
max_depth= 7 | Acurácia (validação): 0.7823
max_depth= 8 | Acurácia (validação): 0.7792
max_depth= 9 | Acurácia (validação): 0.7823
max_depth=10 | Acurácia (validação): 0.7745

Melhor Árvore de Decisão: max_depth=3 | Acurácia=0.7854


### 3.2 Random Forest

In [21]:
best_rf_accuracy = 0
best_rf_estimators = 0
best_rf_depth = 0

for n_est in [10, 20, 40, 60, 80, 100]:
    for depth in [3, 5, 7, 9, None]:
        model_rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=12345)
        model_rf.fit(features_train, target_train)
        predictions_valid = model_rf.predict(features_valid)
        accuracy = accuracy_score(target_valid, predictions_valid)
        if accuracy > best_rf_accuracy:
            best_rf_accuracy = accuracy
            best_rf_estimators = n_est
            best_rf_depth = depth
            best_rf_model = model_rf

print(f'Melhor Random Forest: n_estimators={best_rf_estimators} | max_depth={best_rf_depth} | Acurácia={best_rf_accuracy:.4f}')

Melhor Random Forest: n_estimators=40 | max_depth=7 | Acurácia=0.8025


### 3.3 Regressão Logística

In [22]:
model_lr = LogisticRegression(max_iter=1000, random_state=12345)
model_lr.fit(features_train, target_train)
predictions_valid = model_lr.predict(features_valid)
lr_accuracy = accuracy_score(target_valid, predictions_valid)

print(f'Regressão Logística | Acurácia (validação): {lr_accuracy:.4f}')

Regressão Logística | Acurácia (validação): 0.7107


### 3.4 Comparação dos modelos

In [23]:
results = pd.DataFrame({
    'Modelo': ['Árvore de Decisão', 'Random Forest', 'Regressão Logística'],
    'Acurácia (validação)': [best_dt_accuracy, best_rf_accuracy, lr_accuracy]
})
results = results.sort_values('Acurácia (validação)', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

             Modelo  Acurácia (validação)
      Random Forest              0.802488
  Árvore de Decisão              0.785381
Regressão Logística              0.710731


### Conclusões — Comparação de modelos

- O **Random Forest** obteve a melhor acurácia no conjunto de validação, sendo o modelo selecionado para a avaliação final.
- A **Árvore de Decisão** também superou o limiar de 0.75 com profundidades intermédias — profundidades muito altas levam a overfitting.
- A **Regressão Logística** serve como baseline; sendo um modelo linear, captura menos a complexidade dos dados.

O modelo final escolhido é o **Random Forest** com os melhores hiperparâmetros encontrados.

## 4. Avaliação no conjunto de teste

Avaliamos o melhor modelo **apenas uma vez** no conjunto de teste — dados que o modelo nunca viu durante o treino ou a seleção de hiperparâmetros.

In [24]:
# Usar o melhor modelo Random Forest já treinado
predictions_test = best_rf_model.predict(features_test)
test_accuracy = accuracy_score(target_test, predictions_test)

print(f'Acurácia no conjunto de TESTE: {test_accuracy:.4f}')
print()
if test_accuracy >= 0.75:
    print('✅ O modelo atingiu o limiar de acurácia de 0.75!')
else:
    print('❌ O modelo não atingiu o limiar de 0.75.')

Acurácia no conjunto de TESTE: 0.8025

✅ O modelo atingiu o limiar de acurácia de 0.75!


<div class="alert alert-block alert-warning">

<b>Comentário do revisor: </b> <a class="tocSkip"></a>
    
- Correto. Porém, o ideal é testar o melhor modelo de cada tipo no conjunto teste para determinar qual é o vencedor.

</div>

### Conclusões — Teste

O modelo final (Random Forest) foi avaliado no conjunto de teste e atingiu uma acurácia acima de 0.75, confirmando que generaliza bem para dados novos.

A acurácia no conjunto de teste pode ser ligeiramente diferente da validação — algo esperado e normal, pois são amostras distintas.

## 5. Prova real (Sanity Check)

Para verificar que o modelo é genuinamente útil, comparamos a sua acurácia com a de um **classificador ingénuo** — que simplesmente prevê sempre a classe mais frequente.

Se o nosso modelo não for melhor do que este baseline trivial, algo está errado.

In [25]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent', random_state=12345)
dummy.fit(features_train, target_train)
dummy_predictions = dummy.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f'Acurácia do classificador ingénuo: {dummy_accuracy:.4f}')
print(f'Acurácia do nosso modelo:          {test_accuracy:.4f}')
print(f'Melhoria:                          +{test_accuracy - dummy_accuracy:.4f}')

Acurácia do classificador ingénuo: 0.6843
Acurácia do nosso modelo:          0.8025
Melhoria:                          +0.1182


### Conclusões — Prova real

O nosso modelo supera claramente o classificador ingénuo, confirmando que aprendeu padrões reais nos dados.

## Conclusão geral

| Etapa | Resultado |
|---|---|
| Dataset | 3214 registos, sem valores ausentes |
| Divisão | 60% treino / 20% validação / 20% teste |
| Melhor modelo | Random Forest |
| Acurácia (validação) | > 0.79 |
| Acurácia (teste) | > 0.79 |
| Limiar atingido (0.75) | ✅ Sim |
| Supera baseline ingénuo | ✅ Sim |

O modelo de **Random Forest** é a solução recomendada para a Megaline identificar automaticamente o plano mais adequado para cada cliente.